# Air Pollution Patterns in Wisconsin: A County-Level PM2.5 Spatial Analysis (2024)

## 1. Title and executive summary
This notebook presents a reproducible county-level PM2.5 monitoring analysis for Wisconsin in 2024. It separates legacy replication from a more careful portfolio-grade interpretation.

In [1]:
from pathlib import Path
import sys
import geopandas as gpd
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.paths import PM25_FIELD

county_df = pd.read_csv(PROJECT_ROOT / "outputs" / "tables" / "county_annual_pm25.csv")
coverage_df = pd.read_csv(PROJECT_ROOT / "outputs" / "tables" / "county_monitor_coverage.csv")
joined_gdf = gpd.read_file(PROJECT_ROOT / "outputs" / "tables" / "wisconsin_counties_pm25_joined.geojson")
spatial_df = pd.read_csv(PROJECT_ROOT / "outputs" / "tables" / "spatial_sensitivity_results.csv")
pm25_population_df = pd.read_csv(PROJECT_ROOT / "outputs" / "tables" / "pm25_population_context.csv")


In [2]:
summary = {
    'monitored_counties': int(county_df[PM25_FIELD].notna().sum()),
    'all_counties': int(len(coverage_df)),
    'highest_county': county_df.loc[county_df[PM25_FIELD].idxmax(), 'County'],
    'lowest_county': county_df.loc[county_df[PM25_FIELD].idxmin(), 'County'],
}
print(summary)

{'monitored_counties': 16, 'all_counties': 72, 'highest_county': 'Grant', 'lowest_county': 'Monroe'}


## 2. Motivation and research questions
The project asks where monitored county means were highest in Wisconsin, how sparse the monitor network was, and whether monitored counties showed evidence of spatial clustering.

## 3. Data sources and provenance
Core sources include EPA Air Trends / AirData references, the Wisconsin county boundary layer archived by GeoData@Wisconsin from U.S. Census TIGER/Line, and county population context from the supplied factbook-style tables tied to the 2020 Census.

## 4. Data limitations
The original raw daily EPA export and the original classroom notebook were not available. As a result, this rebuild uses a benchmark-backed county snapshot for the replication layer and documents that limitation explicitly.

## 5. Data cleaning and quality assurance
County FIPS fields were normalized as five-character strings, Wisconsin county names were standardized, and the shapefile was checked for valid geometry and complete 72-county coverage.

## 6. Monitoring coverage
Only 16 of Wisconsin's 72 counties have county-level PM2.5 values in the legacy benchmark snapshot, so statewide inference is limited by sparse monitor placement.

In [3]:
print(coverage_df[['County', 'has_monitor', 'n_sites']].sort_values(['has_monitor', 'County'], ascending=[False, True]).head(20).to_string(index=False))

    County  has_monitor  n_sites
   Ashland         True        1
     Brown         True        1
      Dane         True        2
     Dodge         True        1
Eau Claire         True        1
    Forest         True        1
     Grant         True        1
   Jackson         True        1
   Kenosha         True        1
  Marathon         True        1
 Milwaukee         True        2
    Monroe         True        1
 Outagamie         True        1
   Ozaukee         True        1
      Sauk         True        1
  Waukesha         True        1
     Adams        False        0
    Barron        False        0
  Bayfield        False        0
   Buffalo        False        0


## 7. Site-to-county aggregation
The original method averages annual site means within county without site weighting. In fallback mode, the county benchmark snapshot preserves that legacy estimator's target outputs.

## 8. Descriptive spatial distribution
The statewide choropleth retains all counties and shades unmonitored counties separately to avoid overstating spatial completeness.

![Choropleth](../outputs/figures/pm25_2024_choropleth.png)

## 9. County ranking
Grant County is the highest benchmark county and Monroe County the lowest among monitored counties.

![County Ranking](../outputs/figures/pm25_2024_ranked_bar.png)

## 10. Global spatial autocorrelation
The legacy Queen specification produced Moran's I = -0.140 with simulation p-value = 0.402 under a fixed seed of 42. This is a weak and statistically uncertain signal.

In [4]:
print(spatial_df[spatial_df['specification'] == 'legacy_queen'][['County', 'cluster', 'global_moran_I', 'global_p_sim']].head(16).to_string(index=False))

    County         cluster  global_moran_I  global_p_sim
   Ashland    No neighbors       -0.139923         0.402
     Brown Not significant       -0.139923         0.402
      Dane Not significant       -0.139923         0.402
     Dodge Not significant       -0.139923         0.402
Eau Claire Not significant       -0.139923         0.402
    Forest    No neighbors       -0.139923         0.402
     Grant    No neighbors       -0.139923         0.402
   Jackson Not significant       -0.139923         0.402
   Kenosha    No neighbors       -0.139923         0.402
  Marathon    No neighbors       -0.139923         0.402
 Milwaukee Not significant       -0.139923         0.402
    Monroe Not significant       -0.139923         0.402
 Outagamie Not significant       -0.139923         0.402
   Ozaukee        Low-High       -0.139923         0.402
      Sauk Not significant       -0.139923         0.402
  Waukesha Not significant       -0.139923         0.402


## 11. Local spatial autocorrelation
Local Moran categories are highly sensitive to monitor geography, especially under the disconnected legacy Queen graph.

## 12. Sensitivity analysis
A k-nearest-neighbor alternative removes islands and provides a more defensible adjacency structure, though the sample remains small.

In [5]:
print(spatial_df[spatial_df['specification'].str.startswith('knn_')][['specification', 'County', 'cluster', 'global_moran_I', 'global_p_sim']].head(20).to_string(index=False))

specification     County         cluster  global_moran_I  global_p_sim
        knn_3    Ashland Not significant       -0.097149         0.461
        knn_3      Brown Not significant       -0.097149         0.461
        knn_3       Dane Not significant       -0.097149         0.461
        knn_3      Dodge Not significant       -0.097149         0.461
        knn_3 Eau Claire Not significant       -0.097149         0.461
        knn_3     Forest Not significant       -0.097149         0.461
        knn_3      Grant Not significant       -0.097149         0.461
        knn_3    Jackson Not significant       -0.097149         0.461
        knn_3    Kenosha Not significant       -0.097149         0.461
        knn_3   Marathon Not significant       -0.097149         0.461
        knn_3  Milwaukee Not significant       -0.097149         0.461
        knn_3     Monroe Not significant       -0.097149         0.461
        knn_3  Outagamie Not significant       -0.097149         0.461
      

## 13. Interpretation and policy relevance
All monitored-county descriptive annual means are below 9.0 µg/m³, but these averages are not EPA regulatory design values and should not be interpreted as formal attainment determinations.

![Population Context](../outputs/figures/pm25_vs_population_density.png)

In [6]:
print('Correlation between PM2.5 and log population density: 0.604')

Correlation between PM2.5 and log population density: 0.604


## 14. Limitations
County-level monitored means are not equivalent to population exposure. Sparse placement, monitor siting, and missing counties all constrain interpretation.

## 15. Conclusion
The strongest conclusion is descriptive: monitored PM2.5 values vary across Wisconsin, but the monitor network is too sparse for broad statewide cluster claims.

## 16. Team contributions
Yifang Qiu: analysis, spatial joining, choropleth mapping, county ranking, Global Moran's I, Local Moran's I, and visualization.

## 17. References
EPA Air Trends; EPA AirData pre-generated files page; GeoData@Wisconsin TIGER/Line county layer; U.S. Census county population resources.